In [126]:
import numpy as np
import pandas as pd
import os

import cv2
from skimage.feature import hog
from skimage import exposure
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

In [127]:
df = pd.read_csv('image_metadata.csv')
df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,0,P. aeruginosa
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,1,P. aeruginosa
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,2,P. aeruginosa
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,3,P. aeruginosa
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,4,P. aeruginosa
...,...,...,...,...,...,...,...,...
106838,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,25,E. coli
106839,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,26,E. coli
106840,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,27,E. coli
106841,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,28,E. coli


In [128]:
df['experiment_id'].value_counts()

experiment_id
BV6134 AST FISH 201126    23340
BV6126 AST FISH 201116    20434
BV6129 AST FISH 201117    17205
BV6128 AST FISH 201117    14490
BV6127 AST FISH 201117    14016
BV6131 AST FISH 201124    12493
BV6133 AST FISH 201125     4865
Name: count, dtype: int64

Taking last 2 experiments, 131 and 133 as test dataset

In [129]:
filtered_df = df[(df['species_label'] != 'Unknown') & #only working with known species
                 (df['treated'] == 'Untreated')] # &  # removed treated antibiotic samples since the later timepoint images are harder to use
                 #(df['timepoint'] == 15)] # looking at frame 15 for the 30 minute point
filtered_df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,0,P. aeruginosa
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,1,P. aeruginosa
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,2,P. aeruginosa
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,3,P. aeruginosa
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,4,P. aeruginosa
...,...,...,...,...,...,...,...,...
100838,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,181,14,5783-14,Untreated,25,E. faecalis
100839,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,181,14,5783-14,Untreated,26,E. faecalis
100840,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,181,14,5783-14,Untreated,27,E. faecalis
100841,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,181,14,5783-14,Untreated,28,E. faecalis


In [130]:
filtered_df['species_label'].value_counts()

species_label
K. pneumoniae    39888
E. coli          18008
E. faecalis      11767
P. aeruginosa     8058
Name: count, dtype: int64

In [131]:
n = 1000
filtered_df_sampled = filtered_df.groupby('species_label').apply(lambda x: x.sample(n=n, random_state=18)).reset_index(drop=True)
# filtered_df_sampled #adjusted to 250 samples per species to account for majority class imbalance
filtered_df_sampled

C:\Users\silve\AppData\Local\Temp\ipykernel_11028\1742201321.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_df_sampled = filtered_df.groupby('species_label').apply(lambda x: x.sample(n=n, random_state=18)).reset_index(drop=True)


,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,120,14,1635-14,Untreated,21,E. coli
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,101,5,343-05,Untreated,9,E. coli
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,109,18,887-01,Untreated,15,E. coli
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,116,24,1243-02,Untreated,10,E. coli
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,147,22,3471-05,Untreated,20,E. coli
...,...,...,...,...,...,...,...,...
3995,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,174,23,5883-01,Untreated,17,P. aeruginosa
3996,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,185,23,6763-01,Untreated,27,P. aeruginosa
3997,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6131 AST FISH 201124,105,39,563-17,Untreated,19,P. aeruginosa
3998,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,187,9,6923-09,Untreated,8,P. aeruginosa


In [132]:
filtered_df_sampled['experiment_id'].value_counts()

experiment_id
BV6134 AST FISH 201126    831
BV6126 AST FISH 201116    706
BV6128 AST FISH 201117    620
BV6127 AST FISH 201117    555
BV6129 AST FISH 201117    532
BV6131 AST FISH 201124    428
BV6133 AST FISH 201125    328
Name: count, dtype: int64

In [133]:
def load_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (128, 128))
    return image

In [134]:
def extract_features(image):
    # HOG Features
    hog_features, _ = hog(image, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    
    # Canny Edge Features
    edges = cv2.Canny(image, 100, 200)
    edge_features = edges.flatten()
    
    # Contour Features
    ret, thresh = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contour_features = []
    
    max_contours = 5  # Limit to first 5 contours
    for contour in contours[:max_contours]:  
        moments = cv2.moments(contour)
        hu_moments = cv2.HuMoments(moments).flatten()
        contour_features.extend(hu_moments)
    
    # Pad if fewer than max_contours are found
    while len(contour_features) < max_contours * 7:
        contour_features.append(0.0)

    contour_features = np.array(contour_features)
    
    # Combine all features
    features = np.hstack([hog_features, edge_features, contour_features])  # Limit edge features to 500
    return features

In [135]:
def load_dataset(image_paths, labels, timepoints, experiment_ids):
    X = []
    y = []
    for img_path, label, time, experiment_id in zip(image_paths, labels, timepoints, experiment_ids):
        image = load_image(img_path)
        features = extract_features(image)
        time = np.array([time*2]) #convert frame to minutes, every frame is 2 minutes
        all_features = np.hstack([features, time])
        X.append(all_features)
        y.append([label, experiment_id])
    return np.array(X), np.array(y)

In [136]:
def get_cols(filtered_df_sampled):    
    img_paths = []
    labels = []
    timepoints = []
    experiment_ids = []
    for _, row in filtered_df_sampled.iterrows():
        timepoints.append(row['timepoint'])  # Timepoint
        img_paths.append(row['file_path']) # Path to image file
        labels.append(row['species_label']) # Target variable
        experiment_ids.append(row['experiment_id']) # For training and testing split
    return img_paths, labels, timepoints, experiment_ids
img_paths, labels, timepoints, experiment_ids = get_cols(filtered_df_sampled)

In [ ]:
species_map = {'E. faecalis': 0, 'K. pneumoniae': 1, 'E. coli': 2, 'P. aeruginosa': 3}
numerical_labels = [species_map[label] for label in labels]

X, y = load_dataset(img_paths, numerical_labels, timepoints, experiment_ids)

test_experiment_ids = np.array(['BV6127 AST FISH 201117', 'BV6129 AST FISH 201117'])
test_indices = np.isin(y[:, -1], test_experiment_ids)  # experiment_id last column

# remove experiment_id from y
y = y[:, 0]

# Split data using boolean masking
X_train = X[~test_indices]
y_train = y[~test_indices]
X_test = X[test_indices]
y_test = y[test_indices]

KeyboardInterrupt: 

In [ ]:
print(X_train.shape)

(2913, 24519)


In [ ]:
print(f"Training set class distribution: {pd.Series(y_train).value_counts()}")
print(f"Test set class distribution: {pd.Series(y_test).value_counts()}")

Training set class distribution: 2    826
3    744
1    698
0    645
Name: count, dtype: int64
Test set class distribution: 0    355
1    302
3    256
2    174
Name: count, dtype: int64


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
clf = SVC(kernel='linear')
clf.fit(X_train, y_train)

SVC(kernel='linear')

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       355
           1       0.93      0.82      0.87       302
           2       0.83      0.93      0.88       174
           3       0.91      0.96      0.94       256

    accuracy                           0.92      1087
   macro avg       0.91      0.92      0.92      1087
weighted avg       0.92      0.92      0.92      1087



In [ ]:
import joblib

In [ ]:
joblib.dump(clf, "svm_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']